<a href="https://colab.research.google.com/github/DiegoMedinaMorais/I1DS-2026/blob/main/TensorFlow_e_Keras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rede Neural

In [8]:
import yfinance as yf
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

dados = yf.download("PETR4.SA", start="2018-01-01")

dados["retorno"] = dados["Close"].pct_change()
dados["media5"] = dados["Close"].rolling(5).mean()
dados["alvo"] = np.where(dados["Close"].shift(-1) > dados["Close"], 1, 0)

dados = dados.dropna()

x = dados[["retorno", "media5"]]
y = dados["alvo"]

corte = int(len(dados) * 0.8)

scaler = StandardScaler()
x_treino = scaler.fit_transform(x[:corte])
x_teste = scaler.transform(x[corte:])

modelo = tf.keras.Sequential([
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
modelo.fit(x_treino, y[:corte], epochs=20, verbose=0)

loss, acc = modelo.evaluate(x_teste, y[corte:], verbose=0)
previsao = modelo.predict(scaler.transform(x.tail(1)), verbose=0)[0][0]

print(f"Acurácia: {acc:.2%}")
print(f"Chance de alta no próximo pregão: {previsao:.2%}")

print("Previsão: ALTA" if previsao > 0.5 else "Previsão: QUEDA")

/tmp/ipykernel_8638/2766236779.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  dados = yf.download("PETR4.SA", start="2018-01-01")
[*********************100%***********************]  1 of 1 completed


Acurácia: 50.36%
Chance de alta no próximo pregão: 50.32%
Previsão: ALTA
